# PolaFusion — Model Training

Cells 1–3: 5-fold mDeBERTa-v3-base training for Subtasks 1, 2, 3  
Cells 4–6: 3-fold XLM-RoBERTa-large training for Subtasks 1, 2, 3  

**Run order:** Run each subtask cell independently on Kaggle with A100 GPU

In [ ]:
#5fold mdeberta training of subtask-1 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 1  # TARGETING SUBTASK 1
MODEL_NAME = "microsoft/mdeberta-v3-base" 
N_FOLDS = 5
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = f"./subtask{TASK_ID}_5fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_final_models"

TARGET_COLS = ["polarization"]

# --- DATA LOADING ---
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"
df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")


# CRITICAL: Drop NaNs but KEEP ZEROS.
df = df.dropna(subset=TARGET_COLS)
df['polarization'] = df['polarization'].astype(int)

# --- METRICS FUNCTION (NEW) ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid to get probability 0-1
    probs = 1 / (1 + np.exp(-logits))
    # Threshold at 0.5
    predictions = (probs > 0.5).astype(int)
    
    # Calculate F1 Macro
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    
    return {"f1_macro": f1, "accuracy": acc}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting 5-Fold Training for Subtask {TASK_ID} with F1 Score...")

for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type="regression"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", # Safer to save based on Loss
        greater_is_better=False,
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics  # <--- Added Metric Calculation
    )
    
    trainer.train()
    
    # Save Clean Model
    final_model_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_model_path)
    print(f"   ✅ Saved Clean Model to: {final_model_path}")
    
    # Delete Heavy Checkpoints
    if os.path.exists(fold_checkpoint_dir):
        shutil.rmtree(fold_checkpoint_dir)
        print(f"   🗑️ DELETED Heavy Checkpoints: {fold_checkpoint_dir}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ 5-Fold Training Complete! Models saved in {FINAL_MODELS_DIR}")

In [ ]:
#5fold mdeberta training of subtask-2 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 2  # TARGETING SUBTASK 2
MODEL_NAME = "microsoft/mdeberta-v3-base" 
N_FOLDS = 5
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
OUTPUT_DIR = f"./subtask{TASK_ID}_5fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_final_models"

# Define Labels for Task 2
TARGET_COLS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]

# --- DATA LOADING ---
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"
df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")


# Filter: Keep ONLY Polarized rows
df = df.dropna(subset=TARGET_COLS)
df['sum'] = df[TARGET_COLS].sum(axis=1)
df = df[df['sum'] > 0].reset_index(drop=True)

# --- STRATIFICATION STRATEGY (CRITICAL FIX) ---
# Create a string representing the label combination (e.g. "0_1_0_0_1")
# This treats unique label sets as distinct classes for stratification
df['label_str'] = df[TARGET_COLS].astype(int).astype(str).agg('_'.join, axis=1)

# Combine with Language to ensure language balance across folds
# We limit the stratification granularity to avoid "class with 1 member" errors
# If a specific combo is too rare, we stratify just by Label Sum + Language
df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

# --- METRICS (CRITICAL FIX) ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid for multi-label
    probs = 1 / (1 + np.exp(-logits))
    # Use 0.5 threshold for monitoring (we optimize this later)
    predictions = (probs > 0.5).astype(int)
    
    score = f1_score(labels, predictions, average='macro')
    return {"f1_macro": score}

# --- CUSTOM LOSS FUNCTION (FOCAL LOSS) ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET CLASS ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Use StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting Stratified 5-Fold Training for Subtask {TASK_ID}...")
print(f"   Target Metric: F1 Macro (Higher is better)")

# We split using the 'stratify_key' we created earlier
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_key'])):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    # Clean previous temp folder
    if fold > 0:
        prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
        if os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=len(TARGET_COLS), 
        problem_type="multi_label_classification"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        
        # METRICS & SAVING
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",  # <--- CRITICAL FIX: Optimize F1
        greater_is_better=True,            # <--- CRITICAL FIX: Higher is better
        
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics    # <--- CRITICAL FIX: Monitor F1
    )
    
    trainer.train()
    
    # Save Final Clean Model
    final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_path)
    print(f"   ✅ Saved Best F1 Model to: {final_path}")
    
    # Delete Heavy Checkpoints
    if os.path.exists(fold_checkpoint_dir):
        shutil.rmtree(fold_checkpoint_dir)
        print(f"   🗑️ DELETED Heavy Checkpoints: {fold_checkpoint_dir}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ 5-Fold Stratified Training Complete! Models in {FINAL_MODELS_DIR}")

In [ ]:
#5fold mdeberta training of subtask-3 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 3  # TARGETING SUBTASK 2
MODEL_NAME = "microsoft/mdeberta-v3-base" 
N_FOLDS = 5
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
OUTPUT_DIR = f"./subtask{TASK_ID}_5fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_final_models"

# Define Labels for Task 3
TARGET_COLS = ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"]

# --- DATA LOADING ---
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"
df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")


# Filter: Keep ONLY Polarized rows
df = df.dropna(subset=TARGET_COLS)
df['sum'] = df[TARGET_COLS].sum(axis=1)
df = df[df['sum'] > 0].reset_index(drop=True)

# --- STRATIFICATION STRATEGY (CRITICAL FIX) ---
# Create a string representing the label combination (e.g. "0_1_0_0_1")
# This treats unique label sets as distinct classes for stratification
df['label_str'] = df[TARGET_COLS].astype(int).astype(str).agg('_'.join, axis=1)

# Combine with Language to ensure language balance across folds
# We limit the stratification granularity to avoid "class with 1 member" errors
# If a specific combo is too rare, we stratify just by Label Sum + Language
df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

# --- METRICS (CRITICAL FIX) ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid for multi-label
    probs = 1 / (1 + np.exp(-logits))
    # Use 0.5 threshold for monitoring (we optimize this later)
    predictions = (probs > 0.5).astype(int)
    
    score = f1_score(labels, predictions, average='macro')
    return {"f1_macro": score}

# --- CUSTOM LOSS FUNCTION (FOCAL LOSS) ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET CLASS ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Use StratifiedKFold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting Stratified 5-Fold Training for Subtask {TASK_ID}...")
print(f"   Target Metric: F1 Macro (Higher is better)")

# We split using the 'stratify_key' we created earlier
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_key'])):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    # Clean previous temp folder
    if fold > 0:
        prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
        if os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=len(TARGET_COLS), 
        problem_type="multi_label_classification"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        
        # METRICS & SAVING
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",  # <--- CRITICAL FIX: Optimize F1
        greater_is_better=True,            # <--- CRITICAL FIX: Higher is better
        
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics    # <--- CRITICAL FIX: Monitor F1
    )
    
    trainer.train()
    
    # Save Final Clean Model
    final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_path)
    print(f"   ✅ Saved Best F1 Model to: {final_path}")
    
    # Delete Heavy Checkpoints
    if os.path.exists(fold_checkpoint_dir):
        shutil.rmtree(fold_checkpoint_dir)
        print(f"   🗑️ DELETED Heavy Checkpoints: {fold_checkpoint_dir}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ 5-Fold Stratified Training Complete! Models in {FINAL_MODELS_DIR}")

In [ ]:
#3fold xlmr training of subtask-1 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 1  
MODEL_NAME = "xlm-roberta-large" 
N_FOLDS = 3  
MAX_LEN = 128

# --- MEMORY SETTINGS ---
BATCH_SIZE = 8                   
GRADIENT_ACCUMULATION = 2        
EPOCHS = 3                       
LEARNING_RATE = 1e-5             
OUTPUT_DIR = f"./subtask{TASK_ID}_xlmr_large_3fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_xlmr_large_final_models"

# Define Target
TARGET_COLS = ["polarization"]

# --- DATA LOADING ---
# --- DATA LOADING ---
df = pd.read_csv("/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL/subtask1_full_train.csv")

# Drop NaNs but KEEP ZEROS.
df = df.dropna(subset=TARGET_COLS)
df['polarization'] = df['polarization'].astype(int)

# --- ROBUST STRATIFICATION ---
df['stratify_key'] = df['language'] + "_" + df['polarization'].astype(str)
le = LabelEncoder()
df['stratify_label'] = le.fit_transform(df['stratify_key'])

# --- METRICS ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {"f1_macro": f1, "accuracy": acc}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- CLEANUP START ---
# Force clean everything to start fresh
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
if os.path.exists(FINAL_MODELS_DIR):
    shutil.rmtree(FINAL_MODELS_DIR) # Deletes old Fold 0 too, since you want to redo all
print("🧹 Workspace wiped clean.")

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting FRESH XLM-R LARGE Training (All 3 Folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_label'])):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=1, 
        problem_type="regression"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,       
        gradient_accumulation_steps=GRADIENT_ACCUMULATION, 
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        
        # --- SAFETY SETTINGS ---
        # 1. Evaluate to see F1 Score progress
        eval_strategy="epoch",
        
        # 2. DO NOT SAVE CHECKPOINTS
        save_strategy="no",        
        
        # 3. Save only the final state manually
        load_best_model_at_end=False, 
        
        fp16=True, 
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Save Final Model Manually
    final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_path)
    print(f"   ✅ Saved Fold {fold} Model to: {final_path}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ All 3 Folds Complete! Models in {FINAL_MODELS_DIR}")

In [ ]:
#3fold xlmr training of subtask-2 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder  # <--- FIXED: Added LabelEncoder
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 2  # CHANGE TO 3 AFTER THIS FINISHES
MODEL_NAME = "xlm-roberta-large" 
N_FOLDS = 3
MAX_LEN = 128

# --- CRITICAL MEMORY & DISK SETTINGS ---
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EPOCHS = 4
LEARNING_RATE = 1e-5
OUTPUT_DIR = f"./subtask{TASK_ID}_xlmr_large_3fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_xlmr_large_final_models"

# Define Labels
if TASK_ID == 2:
    TARGET_COLS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
elif TASK_ID == 3:
    TARGET_COLS = ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"]

# --- DATA LOADING ---
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"
df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")

# Filter: Keep ONLY Polarized rows
df = df.dropna(subset=TARGET_COLS)
df['sum'] = df[TARGET_COLS].sum(axis=1)
df = df[df['sum'] > 0].reset_index(drop=True)

# --- ROBUST STRATIFICATION (FIXED) ---
# 1. Create string key
df['label_str'] = df[TARGET_COLS].astype(int).astype(str).agg('_'.join, axis=1)
df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

# 2. Encode to Integers (Safer for SKLearn)
le = LabelEncoder()
df['stratify_label'] = le.fit_transform(df['stratify_key'])

# --- METRICS ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    score = f1_score(labels, predictions, average='macro')
    return {"f1_macro": score}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting XLM-R LARGE 3-Fold Training for Subtask {TASK_ID}...")

# FIXED: Passing the integer encoded labels
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_label'])):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    # Clean previous fold to free space
    if fold > 0:
        prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
        if os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"   🧹 Cleaned up previous checkpoint dir: {prev_dir}")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=len(TARGET_COLS), 
        problem_type="multi_label_classification"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        
        # --- DISK SPACE SAVING ---
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_only_model=True,  # CRITICAL: Don't save optimizer state
        
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,  # <--- FIXED: Using tokenizer instead of processing_class
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Save CLEAN Model (Permanent)
    final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_path)
    print(f"   ✅ Saved Best XLM-R Large Model to: {final_path}")
    
    # IMMEDIATE CLEANUP
    if os.path.exists(fold_checkpoint_dir):
        try:
            shutil.rmtree(fold_checkpoint_dir)
            print(f"   🗑️ DELETED Temp Checkpoints: {fold_checkpoint_dir}")
        except Exception as e:
            print(f"   ⚠️ Cleanup Warning: {e}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ 3-Fold XLM-R Large Training Complete! Models in {FINAL_MODELS_DIR}")

In [ ]:
#3fold xlmr training of subtask-3 with augmentation
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder  # <--- FIXED: Added LabelEncoder
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# --- CONFIGURATION ---
TASK_ID = 3  # CHANGE TO 3 AFTER THIS FINISHES
MODEL_NAME = "xlm-roberta-large" 
N_FOLDS = 3
MAX_LEN = 128

# --- CRITICAL MEMORY & DISK SETTINGS ---
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EPOCHS = 4
LEARNING_RATE = 1e-5
OUTPUT_DIR = f"./subtask{TASK_ID}_xlmr_large_3fold_output"
FINAL_MODELS_DIR = f"./subtask{TASK_ID}_xlmr_large_final_models"

# Define Labels
if TASK_ID == 2:
    TARGET_COLS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
elif TASK_ID == 3:
    TARGET_COLS = ["stereotype", "vilification", "dehumanization", "extreme_language", "lack_of_empathy", "invalidation"]

# --- DATA LOADING ---
data_dir = "/kaggle/input/semeval-combined-dataset-evaluation-phase/SEMEVAL"
df = pd.read_csv(f"{data_dir}/subtask{TASK_ID}_full_train.csv")

# Filter: Keep ONLY Polarized rows
df = df.dropna(subset=TARGET_COLS)
df['sum'] = df[TARGET_COLS].sum(axis=1)
df = df[df['sum'] > 0].reset_index(drop=True)

# --- ROBUST STRATIFICATION (FIXED) ---
# 1. Create string key
df['label_str'] = df[TARGET_COLS].astype(int).astype(str).agg('_'.join, axis=1)
df['stratify_key'] = df['language'] + "_" + df['sum'].astype(str)

# 2. Encode to Integers (Safer for SKLearn)
le = LabelEncoder()
df['stratify_label'] = le.fit_transform(df['stratify_key'])

# --- METRICS ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    score = f1_score(labels, predictions, average='macro')
    return {"f1_macro": score}

# --- CUSTOM LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = FocalLoss(gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- DATASET ---
class TaskDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['text'].astype(str).values
        self.labels = df[TARGET_COLS].values
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LEN)
        item = {key: torch.tensor(val) for key, val in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- MAIN LOOP ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"🚀 Starting XLM-R LARGE 3-Fold Training for Subtask {TASK_ID}...")

# FIXED: Passing the integer encoded labels
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['stratify_label'])):
    print(f"\nTraining Fold {fold + 1}/{N_FOLDS}")
    
    # Clean previous fold to free space
    if fold > 0:
        prev_dir = f"{OUTPUT_DIR}/fold_{fold-1}"
        if os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"   🧹 Cleaned up previous checkpoint dir: {prev_dir}")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    
    train_ds = TaskDataset(train_df, tokenizer)
    val_ds = TaskDataset(val_df, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=len(TARGET_COLS), 
        problem_type="multi_label_classification"
    )
    
    fold_checkpoint_dir = f"{OUTPUT_DIR}/fold_{fold}"
    
    args = TrainingArguments(
        output_dir=fold_checkpoint_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        
        # --- DISK SPACE SAVING ---
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_only_model=True,  # CRITICAL: Don't save optimizer state
        
        fp16=True,
        report_to="none"
    )
    
    trainer = FocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,  # <--- FIXED: Using tokenizer instead of processing_class
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Save CLEAN Model (Permanent)
    final_path = f"{FINAL_MODELS_DIR}/fold_{fold}"
    trainer.save_model(final_path)
    print(f"   ✅ Saved Best XLM-R Large Model to: {final_path}")
    
    # IMMEDIATE CLEANUP
    if os.path.exists(fold_checkpoint_dir):
        try:
            shutil.rmtree(fold_checkpoint_dir)
            print(f"   🗑️ DELETED Temp Checkpoints: {fold_checkpoint_dir}")
        except Exception as e:
            print(f"   ⚠️ Cleanup Warning: {e}")
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ 3-Fold XLM-R Large Training Complete! Models in {FINAL_MODELS_DIR}")